<a href="https://colab.research.google.com/github/gautamkushwaha/MLOps-Gautam_m25csa037/blob/MLDLOPs-Exam2026/Q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 2. Install Required Libraries

First, let's install all the necessary Python libraries in our Colab environment.

In [ ]:
!pip install transformers torch sacrebleu sentencepiece gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.9 MB/s eta 0:00:00


In [ ]:
# Assuming 'input.rtf' and 'output.rtf' have been uploaded to the Colab environment.
# If these are Rich Text Format (RTF) files, ensure they contain plain text or convert them to .txt first.

input_source_file = 'input.rtf'
reference_target_file = 'output.rtf'

print(f"Using uploaded files: {input_source_file} for input and {reference_target_file} for reference.")

Using uploaded files: input.rtf for input and output.rtf for reference.


### 4. Create `translate.py` Script

This cell will create the `translate.py` file with the logic to load the pretrained Hugging Face model and translate the `input.txt` to `output.txt`.

In [ ]:
%%writefile translate.py
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load pretrained model and tokenizer
model_name = "Helsinki-NLP/opus-mt-bn-en"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Input and output file paths
input_file = "input.rtf" # Changed to use uploaded input.rtf
output_file = "output.txt" # This will be the generated translation output

# Read input sentences
with open(input_file, "r", encoding="utf-8") as f:
    sentences = f.readlines()

translated_sentences = []
for sentence in sentences:
    sentence = sentence.strip()
    if not sentence:
        translated_sentences.append("")
        continue
    # Tokenize and translate
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
    outputs = model.generate(**inputs)
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    translated_sentences.append(decoded_output)

# Save translated sentences to output file
with open(output_file, "w", encoding="utf-8") as f:
    for ts in translated_sentences:
        f.write(ts + "\n")

print(f"Translation complete. Translated text saved to {output_file}")

Overwriting translate.py


### 5. Run Translation and Save Output

Now, let's execute the `translate.py` script to perform the translation.

In [ ]:
!python translate.py

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100% 258/258 [00:00<00:00, 845.53it/s, Materializing param=model.shared.weight]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Translation complete. Translated text saved to output.txt


### 6. Evaluate Output using SacreBLEU

Finally, we'll use the `sacrebleu` library to compute the BLEU score between the generated `output.txt` and the `reference.txt`.

In [ ]:
import sacrebleu
import os

# Load translated output and reference text
output_file = "output.txt" # This is the generated translation
reference_file = "output.rtf" # This is the user-provided reference

with open(output_file, "r", encoding="utf-8") as f:
    hypothesis = [line.strip() for line in f.readlines()]

with open(reference_file, "r", encoding="utf-8") as f:
    # sacrebleu expects a list of references, even if only one
    # Each reference itself is a list of segments
    references = [[line.strip() for line in f.readlines()]]

# Compute BLEU score
bleu = sacrebleu.corpus_bleu(hypothesis, references)

print(f"BLEU score: {bleu.score:.2f}")

# Get the first translated statement from the output
if hypothesis:
    print(f"\nFirst translated statement from output.txt:\n{hypothesis[0]}")
else:
    print("\noutput.txt is empty or could not be read.")

BLEU score: 0.08

First translated statement from output.txt:
Urtf1\nsi
